In [ ]:
# import packages
#Random state
a=19 # 42, 13 101 19

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import pyplot

from numpy import mean
from numpy import std
from numpy import absolute
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.linear_model import Lasso

#Speicherpfad muss individuell angepasst werden Excel aus ordner Code Verwenden
df = pd.read_excel ('../data/Data.xlsx')



# drop irrelevant column 
df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)


def numCol(df):
    df_numeric = df.select_dtypes(include=[np.number])
    cols = df_numeric.columns.values
    return cols

# apply function
numeric_cols= numCol(df)


# drop all column containing only one unique value
# find single unique value column
col_sv=df.columns[df.nunique() == 1]

# replacing invalid values
df.replace('-', np.nan, inplace=True)

df.drop([ 'Platte', 'Last', 'Stütze', 'c2', 'Esm', 'c1','dg','h', 'Lasteinleitung', 'fym'], axis = 1, inplace=True)
print(df)


In [ ]:
plt.scatter(df['V_Rd'], df['V_test'])
plt.plot(df['V_test'] ,df['V_test'], color='black', linewidth=2)
plt.xlabel('V_Rd_Code [MN]')
plt.ylabel('V_test [MN]')
plt.title('V_Code vs. V_test')
plt.grid(True,linestyle='-',color='0.75')
print('mean SF = ', np.round(np.mean(df['V_test']/df['V_Rd']),2))
plt.savefig('Euro_Code.png')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

model_EC = smf.ols(formula='V_test ~ V_Rd' , data=df).fit()

print(model_EC.summary())

In [ ]:
plt.scatter(df['V_Rd'], df['V_test'], color='blue', label="$EuroCode$")
plt.scatter(df['V_Rd'], df['V_test']/1.19, color='red',label="$SF=1.19$")
plt.plot(df['V_test'] ,df['V_test'], color='black', linewidth=2)
plt.xlabel('V_Rd_Code [MN]')
plt.ylabel('V_test [MN]')
plt.title('V_Code vs. V_test')
plt.grid(True,linestyle='-',color='0.75')
plt.legend()
plt.savefig('Euro_Code_SF.png')

In [ ]:
plt.scatter(df['V_Rd']/1.5, df['V_test'], color='blue', label="$EuroCode$")
plt.scatter(df['V_Rd'], df['V_test']/1.19, color='red',label="$SF=1.19$")
plt.plot(df['V_test'] ,df['V_test'], color='black', linewidth=2)
plt.xlabel('V_Rd_Code [MN]')
plt.ylabel('V_test [MN]')
plt.title('V_Code vs. V_test')
plt.grid(True,linestyle='-',color='0.75')
plt.legend()
plt.savefig('Euro_Code_SF.png')

# Vergleich EC mit KI

In [ ]:
#import MinMaxScaler from sklearn library
from sklearn.preprocessing import MinMaxScaler
# select the properties that should be scaled
cols=['d', 'Fläche', 'rho_l', 'fcmcyl', 'V_test']
# Create Scaler object
scaler = MinMaxScaler(feature_range=(0, 1))
# fit scaler to data
scaler.fit(df[cols])
#scale chosen properties
df_scaled = pd.DataFrame(scaler.transform(df[cols]), columns=cols)


# inspect distribution
print(df_scaled)
print(df)
Max=df.max()
Min=df.min()
print(Max)
print(Min)
v_max=2.681
v_min=0.025

In [ ]:
data = df_scaled.values
X, y = data[:, :-1], data[:, -1]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=a)
df_train, df_test = train_test_split(df_scaled, test_size=0.3,random_state=a)
df_train_V_Rd, df_test_V_Rd = train_test_split(df, test_size=0.3,random_state=a)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

model_1 = smf.ols(formula='V_test ~ d + Fläche  + rho_l  + fcmcyl*d' , data=df_train).fit()

print(model_1.summary())

In [ ]:
yhat_1=model_1.predict(df_test)
error_1=np.sum((yhat_1-df_test['V_test'])**2)/len(yhat_1)
print(error_1)

yhat_1t=model_1.predict(df_train)

plt.scatter(yhat_1  ,y_test, color='blue')
plt.plot(y_test ,y_test, color='black', linewidth=2)

plt.xlabel('V_predicted')
plt.ylabel('V_test')
plt.title('Performance Nonlinear Regression')
plt.grid(True,linestyle='-',color='0.75')

print('MSE %f' %error_1)


In [ ]:
ypred = yhat_1*(v_max-v_min)+v_min
print(ypred)
V_Rd_max=1.83497
V_Rd_min=0.011179
#df_test['V_Rd']=df_test['V_Rd'].apply(lambda x: x*(V_Rd_max-V_Rd_min)+V_Rd_min)
print(df_test_V_Rd['V_Rd'])

In [ ]:
plt.scatter(df_test_V_Rd['V_Rd'],ypred, color='blue', label="EuroCode")
plt.scatter(ypred/1.23  ,df_test_V_Rd['V_Rd'], color='green', label="V_test_split SF=1.23")
plt.plot(ypred ,ypred, color='black', linewidth=2)

plt.xlabel('V_Rd_Code [MN]')
plt.ylabel('V_predicted [MN]')

plt.title('Performance Test_Data on EC')
plt.grid(True,linestyle='-',color='0.75')
plt.legend()


plt.savefig('EC_to_Test_Data.png')

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

model_X = smf.ols(formula='V_test ~ V_Rd' , data=df_test_V_Rd).fit()

print(model_X.summary())